# Cross-Validation & Regularization
## Building Robust Models That Actually Generalize

### CE 315 - Junior Design

**So far we've learned:**
- Linear regression predicts continuous values
- Gradient descent finds optimal parameters
- Logistic regression predicts categories
- Train/test split evaluates models

**Big questions:**
1. Is our train/test split just lucky? How do we **really** know our model is good?
2. How do we prevent overfitting without manually picking features?

**What we'll learn about:**
- **Cross-Validation**: Get reliable performance estimates
- **Regularization**: Prevent overfitting automatically

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# Set style
sns.set_style('whitegrid')
np.random.seed(42)

print("Libraries loaded")

---
# Part 1: The Problem - Is Your Model Really Good?

Let's create some data and see why a single train/test split can be misleading.

In [ ]:
# Generate synthetic data
np.random.seed(42)
n = 100
X = np.random.uniform(0, 10, n).reshape(-1, 1)
y = 2 * X.flatten() + 1 + np.random.normal(0, 2, n)  # y = 2x + 1 + noise

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, s=50)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Our Dataset', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Dataset: {n} samples")

In [ ]:
# Try different random train/test splits
test_scores = []
train_scores = []

print("Testing 20 different random train/test splits:\n")

for i in range(20):
    # Different random split each time
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    train_score = r2_score(y_train, model.predict(X_train))
    test_score = r2_score(y_test, model.predict(X_test))
    
    train_scores.append(train_score)
    test_scores.append(test_score)
 
    print(f"Split {i+1}: Train R² = {train_score:.3f}, Test R² = {test_score:.3f}")

print("...\n")
print(f"Test R² Range: {min(test_scores):.3f} to {max(test_scores):.3f}")
print(f"That's a difference of {max(test_scores) - min(test_scores):.3f}")
print(f"\nMean Test R²: {np.mean(test_scores):.3f} ± {np.std(test_scores):.3f}")

In [ ]:
# Visualize the variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter of train vs test scores
axes[0].scatter(train_scores, test_scores, s=100, alpha=0.6)
axes[0].plot([0.6, 1], [0.6, 1], 'r--', label='Perfect generalization')
axes[0].set_xlabel('Training R²', fontsize=12)
axes[0].set_ylabel('Test R²', fontsize=12)
axes[0].set_title('Training vs Test Performance', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution of test scores
axes[1].hist(test_scores, bins=10, edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(test_scores), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(test_scores):.3f}')
axes[1].set_xlabel('Test R²', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Test Scores', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nProblem: A single train/test split gives you just ONE of these points!")
print("   You might get lucky (high R²) or unlucky (low R²).")
print("   How do you know which one you got?\n")
print("  Solution: Cross-Validation")

---
# Part 2: Cross-Validation - A Better Way to Evaluate

## The Idea: Use ALL Your Data for Both Training and Testing

**K-Fold Cross-Validation:**
1. Split data into K equal parts ("folds")
2. Train K times, each time using a different fold as the test set
3. Average the K test scores

**Benefits:**
- Every data point gets to be in the test set exactly once
- More reliable estimate of model performance
- Reduces variance from random splitting

In [ ]:
# Visualize 5-fold CV
from sklearn.model_selection import KFold

# Create a small example to visualize
n_samples = 25
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(5, 1, figsize=(12, 8))

for fold, (train_idx, test_idx) in enumerate(kf.split(range(n_samples))):
    # Create visualization array
    colors = ['blue'] * n_samples
    for idx in test_idx:
        colors[idx] = 'red'
    
    axes[fold].bar(range(n_samples), [1]*n_samples, color=colors, edgecolor='black')
    axes[fold].set_xlim(-1, n_samples)
    axes[fold].set_ylim(0, 1.5)
    axes[fold].set_ylabel(f'Fold {fold+1}', fontsize=11)
    axes[fold].set_yticks([])
    axes[fold].grid(axis='x', alpha=0.3)
    
    if fold == 0:
        axes[fold].set_title('5-Fold Cross-Validation (Blue=Train, Red=Test)', fontsize=14)
    if fold == 4:
        axes[fold].set_xlabel('Data Points', fontsize=12)

plt.tight_layout()
plt.show()

print("\n How it works:")
print("1. Fold 1: Train on blue, test on red → Score #1")
print("2. Fold 2: Train on blue, test on red → Score #2")
print("3. Fold 3: Train on blue, test on red → Score #3")
print("4. Fold 4: Train on blue, test on red → Score #4")
print("5. Fold 5: Train on blue, test on red → Score #5")
print("\nFinal Score = Average of all 5 scores")
print("\nEvery data point is in the test set exactly once")

In [ ]:
# Implement cross-validation
from sklearn.model_selection import cross_val_score

model = LinearRegression()

# Perform 5-fold CV
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print("5-Fold Cross-Validation Results:\n")
print(f"Fold 1: {cv_scores[0]:.3f}")
print(f"Fold 2: {cv_scores[1]:.3f}")
print(f"Fold 3: {cv_scores[2]:.3f}")
print(f"Fold 4: {cv_scores[3]:.3f}")
print(f"Fold 5: {cv_scores[4]:.3f}")
print("─" * 30)
print(f"Mean: {cv_scores.mean():.3f}")
print(f"Std:  {cv_scores.std():.3f}")

print("\n Compare to our 20 random splits:")
print(f"Random splits mean: {np.mean(test_scores):.3f} ± {np.std(test_scores):.3f}")
print(f"CV estimate:        {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("\nCV gives similar mean but with systematic evaluation")

In [ ]:
# Compare single split vs CV visually
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Single train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_single = LinearRegression()
model_single.fit(X_train, y_train)
single_score = r2_score(y_test, model_single.predict(X_test))

axes[0].scatter(X_train, y_train, alpha=0.6, label='Train', s=50)
axes[0].scatter(X_test, y_test, alpha=0.6, label='Test', s=50, color='red')
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('y', fontsize=12)
axes[0].set_title(f'Single Split (Test R² = {single_score:.3f})', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cross-validation visualization
axes[1].bar(range(1, 6), cv_scores, alpha=0.7, edgecolor='black')
axes[1].axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean = {cv_scores.mean():.3f}')
axes[1].fill_between([-0.5, 5.5], 
                      cv_scores.mean() - cv_scores.std(),
                      cv_scores.mean() + cv_scores.std(),
                      alpha=0.2, color='red', label=f'±1 Std = {cv_scores.std():.3f}')
axes[1].set_xlabel('Fold', fontsize=12)
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('5-Fold Cross-Validation', fontsize=14)
axes[1].set_xticks(range(1, 6))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Point:")
print("Single split gives you one number (could be lucky or unlucky)")
print("CV gives you the mean and variance (reliable estimate)")

### When to Use Different K Values?

**Common choices:**
- **k=5**: Good balance, standard choice
- **k=10**: More thorough, takes longer
- **k=n (Leave-One-Out)**: Maximum thoroughness, very slow

**Rule of thumb:** k=5 or k=10 is almost always fine

---
# Part 3: Overfitting Revisited - The Polynomial Problem

Remember polynomial regression? Let's see how cross-validation helps us detect overfitting.

In [ ]:
# Generate data with non-linear relationship
np.random.seed(42)
X_poly = np.linspace(0, 10, 50).reshape(-1, 1)
y_poly = 0.5 * X_poly.flatten()**2 - 3 * X_poly.flatten() + 10 + np.random.normal(0, 3, 50)

plt.figure(figsize=(10, 6))
plt.scatter(X_poly, y_poly, s=50, alpha=0.6)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Non-linear Data', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Try different polynomial degrees
degrees = [1, 2, 3, 5, 10, 15]
train_scores_poly = []
test_scores_poly = []
cv_scores_poly = []

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_poly, y_poly, test_size=0.2, random_state=42
)

for degree in degrees:
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_p)
    X_test_poly = poly.transform(X_test_p)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train_poly, y_train_p)
    
    # Scores
    train_score = r2_score(y_train_p, model.predict(X_train_poly))
    test_score = r2_score(y_test_p, model.predict(X_test_poly))
    
    # Cross-validation (on full dataset)
    X_poly_full = poly.fit_transform(X_poly)
    # Use shuffled KFold
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_score = cross_val_score(model, X_poly_full, y_poly, cv=cv, scoring='r2').mean()
    
    train_scores_poly.append(train_score)
    test_scores_poly.append(test_score)
    cv_scores_poly.append(cv_score)
    
    print(f"Degree {degree:2d}: Train R²={train_score:.3f}, Test R²={test_score:.3f}, CV R²={cv_score:.3f}")

In [ ]:
# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(degrees, train_scores_poly, 'o-', linewidth=2, markersize=8, label='Training Score')
plt.plot(degrees, test_scores_poly, 's-', linewidth=2, markersize=8, label='Test Score (single split)')
plt.plot(degrees, cv_scores_poly, '^-', linewidth=2, markersize=8, label='CV Score (5-fold)')
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('Polynomial Degree vs Model Performance', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axvline(x=2, color='green', linestyle='--', alpha=0.5, label='Optimal (degree 2)')
plt.show()

print("\n Observations:")
print("1. Training score always increases (can always fit training data better)")
print("2. Test score increases then decreases (overfitting)")
print("3. CV score closely tracks test score (reliable estimate)")
print("\n CV helps us detect overfitting without needing a separate test set")

In [ ]:
# Visualize the fits
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

X_plot = np.linspace(0, 10, 200).reshape(-1, 1)

for i, degree in enumerate(degrees):
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_p)
    X_plot_poly = poly.transform(X_plot)
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train_p)
    y_plot = model.predict(X_plot_poly)
    
    axes[i].scatter(X_train_p, y_train_p, alpha=0.6, s=50, label='Train')
    axes[i].scatter(X_test_p, y_test_p, alpha=0.6, s=50, color='red', label='Test')
    axes[i].plot(X_plot, y_plot, 'g-', linewidth=2, label='Model')
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('y')
    axes[i].set_title(f'Degree {degree} (Test R²={test_scores_poly[i]:.3f})')
    axes[i].set_ylim(-20, 50)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Notice:")
print("Degree 1: Underfitting (too simple)")
print("Degree 2: Just right ")
print("Degree 10-15: Overfitting (wild oscillations between points)")

**The Problem:**
- High-degree polynomials fit training data perfectly
- But they wiggle wildly and fail on new data
- Manually picking the degree is tedious

**The Solution: Regularization**

---
# Part 4: Ridge Regression (L2 Regularization)

## The Big Idea: Penalize Large Weights

**Standard Linear Regression:**
$$\text{Minimize: } MSE = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$$

**Ridge Regression:**
$$\text{Minimize: } MSE + \alpha \sum w_i^2$$

where:
- $\alpha$ = regularization strength (hyperparameter)
- $w_i$ = model weights/coefficients

**What this does:**
- Encourages smaller weights
- Prevents any single feature from dominating
- Reduces model complexity
- Prevents overfitting!

In [ ]:
# Create high-degree polynomial features (will overfit without regularization)
poly = PolynomialFeatures(degree=10)
X_train_poly10 = poly.fit_transform(X_train_p)
X_test_poly10 = poly.transform(X_test_p)

print(f"Original features: {X_train_p.shape[1]}")
print(f"After polynomial transform (degree 10): {X_train_poly10.shape[1]}")
print(f"\nWe now have {X_train_poly10.shape[1]} features to fit on {X_train_p.shape[0]} samples")
print("This is a recipe for overfitting...")

In [ ]:
# Compare regular linear regression vs Ridge with different alphas
alphas = [0, 0.01, 0.1, 1, 10, 100]

print("\nComparing different regularization strengths:\n")
print(f"{'Alpha':<10} {'Train R²':<12} {'Test R²':<12} {'# Features':<15}")
print("─" * 55)

for alpha in alphas:
    if alpha == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=alpha)
    
    model.fit(X_train_poly10, y_train_p)
    
    train_r2 = r2_score(y_train_p, model.predict(X_train_poly10))
    test_r2 = r2_score(y_test_p, model.predict(X_test_poly10))
    
    # Count "effective" features (weights > 0.01)
    n_features = np.sum(np.abs(model.coef_) > 0.01)
    
    print(f"{alpha:<10} {train_r2:<12.3f} {test_r2:<12.3f} {n_features:<15}")

In [ ]:
# Visualize the effect of alpha
alphas_full = np.logspace(-3, 3, 50)  # 0.001 to 1000
train_scores_ridge = []
test_scores_ridge = []

for alpha in alphas_full:
    model = Ridge(alpha=alpha)
    model.fit(X_train_poly10, y_train_p)
    
    train_scores_ridge.append(r2_score(y_train_p, model.predict(X_train_poly10)))
    test_scores_ridge.append(r2_score(y_test_p, model.predict(X_test_poly10)))

# Find optimal alpha
best_idx = np.argmax(test_scores_ridge)
best_alpha = alphas_full[best_idx]
best_score = test_scores_ridge[best_idx]

plt.figure(figsize=(12, 6))
plt.semilogx(alphas_full, train_scores_ridge, 'b-', linewidth=2, label='Training Score')
plt.semilogx(alphas_full, test_scores_ridge, 'r-', linewidth=2, label='Test Score')
plt.axvline(best_alpha, color='green', linestyle='--', linewidth=2, 
            label=f'Optimal α={best_alpha:.3f}')
plt.scatter([best_alpha], [best_score], color='green', s=200, zorder=5, marker='*')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('Ridge Regression: Effect of Regularization', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n Optimal alpha: {best_alpha:.3f}")
print(f"   Test R²: {best_score:.3f}")
print("\n Interpretation:")
print("• Small alpha (left): Little regularization → overfitting")
print("• Medium alpha (middle): Just about right")
print("• Large alpha (right): Too much regularization → underfitting")

In [ ]:
# Visualize how Ridge affects the coefficients
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, alpha in enumerate([0, 1, 100]):
    if alpha == 0:
        model = LinearRegression()
        title = 'No Regularization (α=0)'
    else:
        model = Ridge(alpha=alpha)
        title = f'Ridge (α={alpha})'
    
    model.fit(X_train_poly10, y_train_p)
    
    axes[i].bar(range(len(model.coef_)), model.coef_, alpha=0.7, edgecolor='black')
    axes[i].set_xlabel('Feature Index', fontsize=11)
    axes[i].set_ylabel('Coefficient Value', fontsize=11)
    axes[i].set_title(title, fontsize=12)
    axes[i].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[i].grid(axis='y', alpha=0.3)
    
    # Add max coefficient value as text
    max_coef = np.max(np.abs(model.coef_))
    axes[i].text(0.5, 0.95, f'Max |coef|: {max_coef:.1f}', 
                transform=axes[i].transAxes, fontsize=10, va='top')

plt.tight_layout()
plt.show()

print("\n Notice how Ridge shrinks the coefficients:")
print("   α=0:   Wild, large coefficients (overfitting)")
print("   α=1:   Moderate, controlled coefficients ")
print("   α=100: Tiny coefficients (underfitting)")

---
# Part 5: Lasso Regression (L1 Regularization)

## A Different Kind of Regularization

**Ridge (L2):**
$$\text{Penalty: } \alpha \sum w_i^2$$

**Lasso (L1):**
$$\text{Penalty: } \alpha \sum |w_i|$$

**Key Difference:**
- Ridge shrinks weights toward zero (but rarely exactly zero)
- **Lasso can make weights exactly zero** → automatic feature selection

**When to use:**
- Ridge: When you think most features are useful
- Lasso: When you think many features are irrelevant

In [ ]:
# Compare Ridge vs Lasso
alphas_compare = [0.1, 1, 10]

fig, axes = plt.subplots(len(alphas_compare), 2, figsize=(14, 12))

for i, alpha in enumerate(alphas_compare):
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly10, y_train_p)
    
    axes[i, 0].bar(range(len(ridge.coef_)), ridge.coef_, alpha=0.7, edgecolor='black')
    axes[i, 0].set_ylabel('Coefficient', fontsize=11)
    axes[i, 0].set_title(f'Ridge (α={alpha})', fontsize=12)
    axes[i, 0].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[i, 0].grid(axis='y', alpha=0.3)
    
    n_nonzero_ridge = np.sum(np.abs(ridge.coef_) > 0.01)
    axes[i, 0].text(0.5, 0.95, f'Non-zero: {n_nonzero_ridge}/{len(ridge.coef_)}',
                   transform=axes[i, 0].transAxes, fontsize=10, va='top')
    
    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_poly10, y_train_p)
    
    axes[i, 1].bar(range(len(lasso.coef_)), lasso.coef_, alpha=0.7, 
                  edgecolor='black', color='orange')
    axes[i, 1].set_ylabel('Coefficient', fontsize=11)
    axes[i, 1].set_title(f'Lasso (α={alpha})', fontsize=12)
    axes[i, 1].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[i, 1].grid(axis='y', alpha=0.3)
    
    n_nonzero_lasso = np.sum(np.abs(lasso.coef_) > 0.01)
    axes[i, 1].text(0.5, 0.95, f'Non-zero: {n_nonzero_lasso}/{len(lasso.coef_)}',
                   transform=axes[i, 1].transAxes, fontsize=10, va='top')
    
    if i == len(alphas_compare) - 1:
        axes[i, 0].set_xlabel('Feature Index', fontsize=11)
        axes[i, 1].set_xlabel('Feature Index', fontsize=11)

plt.tight_layout()
plt.show()

print("\n Key Observation:")
print("   Ridge: All coefficients non-zero (just small)")
print("   Lasso: Many coefficients exactly zero ")
print("\n Lasso performs automatic feature selection")

In [ ]:
# Performance comparison
alphas_full = np.logspace(-3, 2, 50)
ridge_test_scores = []
lasso_test_scores = []

for alpha in alphas_full:
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly10, y_train_p)
    ridge_test_scores.append(r2_score(y_test_p, ridge.predict(X_test_poly10)))
    
    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_poly10, y_train_p)
    lasso_test_scores.append(r2_score(y_test_p, lasso.predict(X_test_poly10)))

plt.figure(figsize=(12, 6))
plt.semilogx(alphas_full, ridge_test_scores, 'b-', linewidth=2, label='Ridge (L2)')
plt.semilogx(alphas_full, lasso_test_scores, 'orange', linewidth=2, label='Lasso (L1)')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('Test R² Score', fontsize=12)
plt.title('Ridge vs Lasso Performance', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

best_ridge_idx = np.argmax(ridge_test_scores)
best_lasso_idx = np.argmax(lasso_test_scores)

print(f"\n Best Performance:")
print(f"Ridge:  α={alphas_full[best_ridge_idx]:.3f}, R²={ridge_test_scores[best_ridge_idx]:.3f}")
print(f"Lasso:  α={alphas_full[best_lasso_idx]:.3f}, R²={lasso_test_scores[best_lasso_idx]:.3f}")
print("\nBoth work well - Choice depends on your problem.")

---
# The Bias-Variance Tradeoff

**The fundamental tradeoff in machine learning:**

**Bias (Underfitting):**
- Model is too simple
- Misses important patterns
- High training error AND high test error

**Variance (Overfitting):**
- Model is too complex
- Fits noise in training data
- Low training error BUT high test error

**The sweet spot:**
- Just the right complexity
- Captures real patterns, ignores noise
- Good performance on both training and test

**Regularization helps by:**
- Reducing variance (prevents overfitting)
- At the cost of slightly increased bias
- But total error decreases!

---
#  Practical Guidelines

## When to Use What?

### Cross-Validation
**Always use it for:**
- Model selection (comparing different algorithms)
- Hyperparameter tuning (finding best alpha, etc.)
- Getting reliable performance estimates

**Common K values:**
- k=5: Standard, good balance
- k=10: More thorough, slower

### Regularization
**Use Ridge when:**
- You have many correlated features
- You think most features are useful
- You want stable, robust predictions

**Use Lasso when:**
- You have many features, many irrelevant
- You want automatic feature selection
- You want a sparse model (interpretability)

**Use GridSearchCV to:**
- Find optimal alpha automatically
- Try many hyperparameters systematically
- Save time and effort

## Typical Workflow

1. **Split data**: 80% dev set, 20% final test set (hold out!)
2. **On dev set**: Use CV to compare models and tune hyperparameters
3. **Select best model**: Based on CV scores
4. **Final evaluation**: Test on held-out test set (only once!)
5. **Deploy**: Use model trained on all available data

In [ ]:
# Example: Complete workflow
print("Complete ML Workflow Example\n")
print("="*50)

# 1. Create data and hold out final test set
X_dev, X_final_test, y_dev, y_final_test = train_test_split(
    X_poly, y_poly, test_size=0.2, random_state=42
)
print(f"\n1. Data split:")
print(f"   Development set: {len(X_dev)} samples (for CV)")
print(f"   Final test set:  {len(X_final_test)} samples (hold out!)")

# 2. Prepare features
poly = PolynomialFeatures(degree=10)
X_dev_poly = poly.fit_transform(X_dev)
X_final_test_poly = poly.transform(X_final_test)
print(f"\n2. Feature engineering: {X_dev_poly.shape[1]} polynomial features")

# 3. Use GridSearchCV on dev set to find best model
param_grid = {'alpha': np.logspace(-2, 2, 20)}
grid_search = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
grid_search.fit(X_dev_poly, y_dev)

print(f"\n3. Model selection with 5-fold CV:")
print(f"   Tried {len(param_grid['alpha'])} different alphas")
print(f"   Total fits: {len(param_grid['alpha']) * 5}")
print(f"   Best alpha: {grid_search.best_params_['alpha']:.3f}")
print(f"   Best CV score: {grid_search.best_score_:.3f}")

# 4. Evaluate on final test set (only once!)
final_score = grid_search.score(X_final_test_poly, y_final_test)
print(f"\n4. Final evaluation on held-out test set:")
print(f"   Test R²: {final_score:.3f}")

# 5. Train final model on ALL data for deployment
X_all_poly = poly.fit_transform(X_poly)
final_model = Ridge(alpha=grid_search.best_params_['alpha'])
final_model.fit(X_all_poly, y_poly)

print(f"\n5. Final model trained on all {len(X_poly)} samples")
print(f"   Ready for deployment!")
print("\n" + "="*50)
print("\nFull workflow complete")

---
#  Summary & Key Takeaways

## What We Learned

### Cross-Validation
 **Problem**: Single train/test split is unreliable
 **Solution**: K-fold CV uses all data for training and testing
 **Result**: More reliable performance estimates
 **Use**: Model selection, hyperparameter tuning, evaluation

### Regularization
 **Problem**: Complex models overfit
 **Ridge (L2)**: Shrinks all weights, prevents overfitting
 **Lasso (L1)**: Shrinks weights to exactly zero, feature selection
 **Result**: Better generalization


### Bias-Variance Tradeoff
 **Underfitting**: High bias, too simple
 **Overfitting**: High variance, too complex
 **Regularization**: Reduces variance, slight increase in bias
 **Goal**: Find the sweet spot

## Important Formulas

**Ridge:**
$$\text{Loss} = MSE + \alpha \sum w_i^2$$

**Lasso:**
$$\text{Loss} = MSE + \alpha \sum |w_i|$$

where α controls regularization strength.

## Common Mistakes to Avoid

 Using test set for model selection (use CV instead!)
 Forgetting to scale features before regularization
 Not trying a range of alpha values
 Thinking more features always help (Lasso shows this isn't true)
 Only looking at training error

## What's Next?

-  Feature engineering (pipelines, scaling, encoding)
-  Different models (decision trees, random forests)
-  Use CV to compare models, regularization to prevent overfitting
-  Hyperparameter tuning for nuclear reactor benchmarks


# Possible Practice Problems

## Problem 1: Cross-Validation Comparison
Compare 3-fold, 5-fold, and 10-fold CV on the polynomial data. Do they give similar results? Which has higher variance?

## Problem 2: Regularization Strength
Create a plot showing how the number of non-zero coefficients in Lasso changes with alpha. At what alpha do you have only 3 features left?

## Problem 3: Real Data
Load your student performance data from Assignment 2. Use GridSearchCV to find the best Ridge alpha for predicting final grades. Does regularization improve performance?

## Problem 4: Learning Curves
Create learning curves for Lasso with alpha = [0.01, 0.1, 1, 10]. Which alpha shows the best bias-variance tradeoff?

## Problem 5: Elastic Net
Research Elastic Net (combines Ridge and Lasso). Implement it on the polynomial data. When might you prefer it over pure Ridge or Lasso?

## Bonus Challenge: Pipeline
Create a Pipeline that:
1. Scales features
2. Creates polynomial features
3. Applies Ridge regression

Use GridSearchCV to tune both the polynomial degree AND alpha simultaneously!